In [1]:
# GPU Verification
!nvidia-smi

Thu Mar 19 05:43:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Install core dependencies
!pip install ultralytics boxmot flask flask-cors pyngrok supervision lap cython-bbox
!apt-get install -y ffmpeg

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 7.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.4/217.4 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.9/796.9 kB 49.1 MB/s eta 0:00:00
  Created wheel for cython-bbox: filename=cython_bb

In [3]:
# Create project structure if running locally instead of cloning
import os
os.makedirs('/content/templates', exist_ok=True)
os.makedirs('/content/uploads', exist_ok=True)
os.makedirs('/content/outputs', exist_ok=True)

# If you have analyzer.py, processor.py, app.py, and templates/index.html uploaded to /content,
# we are ready. If not, make sure they are uploaded securely before continuing.

In [4]:
%%writefile analyzer.py
import cv2
import numpy as np

class VideoIntelligenceAnalyzer:
    """
    Intelligently analyzes video properties by sampling frames.
    Determines optimal model and tracking configurations based on
    video resolution, FPS, lighting conditions, and length.
    """
    def __init__(self, video_path):
        self.video_path = video_path

    def analyze(self):
        cap = cv2.VideoCapture(self.video_path)
        if not cap.isOpened():
            return {"error": "Could not open video file"}

        fps = cap.get(cv2.CAP_PROP_FPS)
        if fps <= 0:
            fps = 30.0  # Safe fallback

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        duration = total_frames / fps if fps > 0 else 0

        # Handle empty/invalid videos
        if total_frames == 0 or duration < 0.1:
            cap.release()
            return {"error": "Video is empty or unreadable"}

        # Sample up to 30 evenly spaced frames
        num_samples = min(30, total_frames)
        sample_indices = np.linspace(0, total_frames - 1, num_samples, dtype=int)

        brightness_list = []
        blur_list = []

        for idx in sample_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if not ret:
                continue

            # Convert to grayscale for analysis
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

            # Brightness (average pixel intensity)
            brightness_list.append(np.mean(gray))

            # Motion blur score (variance of the Laplacian)
            blur_score = cv2.Laplacian(gray, cv2.CV_64F).var()
            blur_list.append(blur_score)

        cap.release()

        avg_brightness = np.mean(brightness_list) if brightness_list else 127
        avg_blur = np.mean(blur_list) if blur_list else 1000

        # Determine Resolution Category & imgsz
        pixels = width * height
        resolution_cat = "SD"
        imgsz = 640
        if pixels >= 3840 * 2160 * 0.9:
            resolution_cat = "4K"
            imgsz = 1280
        elif pixels >= 1920 * 1080 * 0.9:
            resolution_cat = "FHD"
            imgsz = 1280
        elif pixels >= 1280 * 720 * 0.9:
            resolution_cat = "HD"
            imgsz = 960

        # Determine Scene Density & Lighting
        scene_type = "Normal"
        conf_threshold = 0.45
        if avg_brightness < 60:
            scene_type = "Dark/Night"
            conf_threshold = 0.25  # Lower threshold to detect dark vehicles
        elif avg_brightness > 200:
            scene_type = "Very Bright"
            conf_threshold = 0.35

        # Motion blur considerations
        motion_blur = "Low Blur"
        if avg_blur < 50:
            motion_blur = "High Blur"
            conf_threshold = max(0.25, conf_threshold - 0.1) # Be more lenient if blurry

        # Intelligent Model Selection
        # On a T4 (16GB), we prioritize yolov8l or yolov8x.
        # 'yolov8x' for shorter or lower FPS videos to maximize accuracy where time permits
        if fps < 15 or duration < 60:
            model_variant = 'yolov8x'
        else:
            if resolution_cat in ["FHD", "4K"]:
                 model_variant = 'yolov8l' # Balance VRAM/speed for high-res
            else:
                 model_variant = 'yolov8x'

        # Frame Skipping for long videos to maintain sensible processing times
        # Avoid skipping on very short clips
        if duration < 60:
            frame_skip = 0
        elif duration <= 300:
            frame_skip = 1
        else:
            frame_skip = 2 # Process every 3rd frame

        # Virtual counting line placement
        # E.g., assume overhead mix and place at ~65%
        counting_line_pos = 0.65

        # Generate human-readable explanation
        explanation = []
        explanation.append(f"• Analyzed {resolution_cat} video ({width}x{height} @ {fps:.1f} FPS, {duration:.1f}s). Optimized inference resolution to {imgsz}px.")
        explanation.append(f"• Lighting detected as '{scene_type}' (brightness: {avg_brightness:.1f}). Motion blur score: {avg_blur:.1f} ({motion_blur}).")
        explanation.append(f"• Auto-selected {model_variant} model to maximize accuracy. Confidence threshold set to {conf_threshold:.2f} due to scene traits.")
        if frame_skip > 0:
            explanation.append(f"• Video is long (>1 min). Enabled frame skipping (skip {frame_skip}) to ensure timely processing without losing tracking integrity.")
        else:
            explanation.append(f"• Processing every frame for maximum tracking precision.")
        explanation.append(f"• Auto-placed virtual counting line at {int(counting_line_pos*100)}% of frame height.")

        # Cap processing time estimation (heuristic: T4 does ~15-20 FPS on YOLOv8l/x, factoring in skip)
        estimated_fps = 18.0 if model_variant == 'yolov8x' else 25.0
        active_frames = total_frames / (frame_skip + 1)
        estimated_processing_time = active_frames / estimated_fps

        config = {
            "resolution": resolution_cat,
            "width": width,
            "height": height,
            "fps": fps,
            "duration": float(duration),
            "total_frames": total_frames,
            "avg_brightness": float(avg_brightness),
            "blur_score": float(avg_blur),
            "conf_threshold": float(conf_threshold),
            "iou_threshold": 0.45,
            "imgsz": imgsz,
            "model_variant": model_variant,
            "tracker": "bytetrack",
            "frame_skip": frame_skip,
            "counting_line_pos": float(counting_line_pos),
            "estimated_time_sec": float(estimated_processing_time)
        }

        return {
            "config": config,
            "explanation": "\n".join(explanation)
        }

Writing analyzer.py


In [5]:
%%writefile processor.py
import cv2
import numpy as np
import os
import json
import csv
import subprocess
from datetime import datetime
from collections import defaultdict
from ultralytics import YOLO

try:
    from boxmot import ByteTrack
except ImportError:
    class ByteTrack:
        def __init__(self, *args, **kwargs): pass

class TrafficProcessor:
    def __init__(self, video_path, config, job_id, progress_store):
        self.video_path = video_path
        self.config = config
        self.job_id = job_id
        self.progress_store = progress_store

        self.output_dir = os.path.join(os.getcwd(), 'outputs', self.job_id)
        os.makedirs(self.output_dir, exist_ok=True)
        self.raw_output_path = os.path.join(self.output_dir, 'raw_output.avi')
        self.final_output_path = os.path.join(self.output_dir, 'output.mp4')

        # Load YOLO Model
        model_name = self.config.get('model_variant', 'yolov8l') + '.pt'
        self.model = YOLO(model_name)

        # Class filter enforcement: strictly allow only COCO class IDs
        # [2=car, 3=motorcycle, 5=bus, 7=truck] plus [1=bicycle].
        self.class_whitelist = [1, 2, 3, 5, 7]
        self.class_names = {1: 'Bicycle', 2: 'Car', 3: 'Motorcycle', 5: 'Bus', 7: 'Truck'}
        self.colors = {
            1: (255, 255, 0),    # Cyan
            2: (255, 0, 0),      # Blue
            3: (0, 255, 0),      # Green
            5: (0, 165, 255),    # Orange
            7: (0, 0, 255)       # Red
        }

        # Tracker Initialization (Tuned for stability)
        self.tracker = ByteTrack(
            track_thresh=0.45,
            track_buffer=60,
            match_thresh=0.85,
            frame_rate=self.config.get('fps', 30)
        )

        self.track_history = defaultdict(lambda: {
            "class_id": None,
            "first_seen": None,
            "last_seen": None,
            "bboxes": [],
            "centroids": [],
            "speeds": [],
            "direction": "UNKNOWN",
            "confidence_avg": 0.0,
            "conf_list": []
        })

        # BUG FIX #1: Unique registries
        self.counted_ids = set()   # for confirmed unique vehicles (min_track_frames)
        self.crossed_ids = set()   # for line-crossing deduplication

        self.vehicles_in = 0
        self.vehicles_out = 0
        self.peak_density = 0

        self.fps = self.config.get('fps', 30)
        self.width = self.config.get('width', 1920)
        self.height = self.config.get('height', 1080)
        self.total_frames = self.config.get('total_frames', 1)

        # Interval tracking for timeline analysis (Feature: Graphical Insights)
        self.interval_sec = 10
        self.interval_frames = self.fps * self.interval_sec
        self.interval_vehicles = defaultdict(set)
        self.class_interval_vehicles = defaultdict(lambda: defaultdict(set))

        # Spatial Heatmap Grid
        self.heatmap_rows = 20
        self.heatmap_cols = 40
        self.spatial_heatmap = np.zeros((self.heatmap_rows, self.heatmap_cols), dtype=int)

        # Counting Line
        line_pct = self.config.get('counting_line_pos', 0.65)
        self.counting_y = int(self.height * line_pct)

        # Speed params
        self.ppm = self.width / 15.0

    def calculate_speed(self, points, fps):
        # BUG FIX #2(d): smoothing with a rolling window of 10 frames
        if len(points) < 10:
            return 0.0
        p1 = points[-10]
        p2 = points[-1]
        dist_pixels = np.linalg.norm(np.array(p1) - np.array(p2))
        dist_meters = dist_pixels / self.ppm
        time_seconds = 9.0 / fps # 9 frame intervals between 10 points
        speed_mps = dist_meters / time_seconds
        speed_kmh = speed_mps * 3.6
        return min(speed_kmh, 180.0)

    def update_progress(self, current_frame):
        if current_frame % max(1, int(self.fps)) == 0 or current_frame == self.total_frames:
            pct = int((current_frame / max(1, self.total_frames)) * 100)
            elapsed = (datetime.now() - self.start_time).total_seconds()
            current_fps = current_frame / max(0.1, elapsed)
            rem_frames = self.total_frames - current_frame
            eta = rem_frames / current_fps if current_fps > 0 else 0

            self.progress_store[self.job_id] = {
                "frames_done": current_frame,
                "total_frames": self.total_frames,
                "percent": pct,
                "current_fps": round(current_fps, 1),
                "eta_seconds": int(eta),
                "status": "processing",
                "vehicles_detected": len(self.counted_ids) # BUG FIX #1: show unique counts
            }

    def process(self):
        self.start_time = datetime.now()
        cap = cv2.VideoCapture(self.video_path)
        fourcc = cv2.VideoWriter_fourcc(*'XVID')
        out = cv2.VideoWriter(self.raw_output_path, fourcc, self.fps, (self.width, self.height))

        frame_idx = 0
        skip = self.config.get('frame_skip', 0)

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            frame_idx += 1
            if skip > 0 and frame_idx % (skip + 1) != 0:
                continue

            results = self.model(
                frame,
                conf=self.config.get('conf_threshold', 0.45),
                iou=self.config.get('iou_threshold', 0.45),
                imgsz=self.config.get('imgsz', 1280),
                classes=self.class_whitelist,
                half=True,
                verbose=False
            )[0]

            dets = []
            for box in results.boxes:
                conf = float(box.conf[0].cpu().numpy())
                # BUG FIX #2(a): Confidence floor
                if conf < 0.30:
                    continue

                cls = int(box.cls[0].cpu().numpy())
                # BUG FIX #2(b): Class filter enforcement
                if cls not in self.class_whitelist:
                    continue

                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                # BUG FIX #2(c): BB size sanity check
                if (x2 - x1) < 20 or (y2 - y1) < 20:
                    continue

                dets.append([x1, y1, x2, y2, conf, cls])

            dets = np.array(dets)
            if len(dets) == 0:
                dets = np.empty((0, 6))

            tracks = self.tracker.update(dets, frame)

            interval_idx = int(frame_idx // self.interval_frames)
            currently_visible = 0
            current_speeds = []

            # Peak density tracking
            if len(tracks) > self.peak_density:
                self.peak_density = len(tracks)

            cv2.line(frame, (0, self.counting_y), (self.width, self.counting_y), (0, 255, 255), 2)
            cv2.putText(frame, f"IN: {self.vehicles_in} | OUT: {self.vehicles_out}",
                        (20, self.counting_y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

            if len(tracks) > 0:
                for track in tracks:
                    x1, y1, x2, y2, track_id, conf, cls, _ = track
                    x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
                    cls = int(cls)
                    track_id = int(track_id)

                    cx = int((x1 + x2) / 2)
                    cy = int((y1 + y2) / 2)

                    # Accumulate presence in spatial heatmap
                    grid_c = min(int((cx / self.width) * self.heatmap_cols), self.heatmap_cols - 1)
                    grid_r = min(int((cy / self.height) * self.heatmap_rows), self.heatmap_rows - 1)
                    self.spatial_heatmap[grid_r, grid_c] += 1

                    hist = self.track_history[track_id]
                    if hist['class_id'] is None:
                        hist['class_id'] = cls
                        hist['first_seen'] = frame_idx
                        hist['bboxes'].append((x1, y1, x2, y2))

                    hist['last_seen'] = frame_idx
                    hist['centroids'].append((cx, cy))
                    hist['conf_list'].append(conf)

                    currently_visible += 1

                    # BUG FIX #1: Unique Vehicle Registry (min frames = 5)
                    if len(hist['centroids']) >= 5:
                        self.counted_ids.add(track_id)
                        cname = self.class_names.get(hist['class_id'], 'Unknown').lower()
                        self.interval_vehicles[interval_idx].add(track_id)
                        self.class_interval_vehicles[cname][interval_idx].add(track_id)

                    # BUG FIX #1: Line-Crossing Deduplication
                    if len(hist['centroids']) >= 2:
                        prev_y = hist['centroids'][-2][1]
                        if track_id not in self.crossed_ids:
                            # Moving UP / OUT -> Out implies moving generally down if overhead?
                            # Let's use standard direction logic
                            if prev_y < self.counting_y and cy >= self.counting_y:
                                self.vehicles_out += 1
                                self.crossed_ids.add(track_id)
                                hist['direction'] = "OUT"
                            elif prev_y > self.counting_y and cy <= self.counting_y:
                                self.vehicles_in += 1
                                self.crossed_ids.add(track_id)
                                hist['direction'] = "IN"

                    # Speed Calculation
                    speed_kmh = self.calculate_speed(hist['centroids'], self.fps)
                    if speed_kmh > 0:
                        hist['speeds'].append(speed_kmh)
                        current_speeds.append(speed_kmh)

                    disp_speed = hist['speeds'][-1] if hist['speeds'] else 0.0

                    # Render Box
                    color = self.colors.get(cls, (255, 255, 255))
                    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

                    label_text = f"#{track_id} {self.class_names.get(cls, 'Unknown')} | {disp_speed:.1f} km/h"
                    (tw, th), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
                    cv2.rectangle(frame, (x1, y1 - 20), (x1 + tw, y1), color, -1)
                    cv2.putText(frame, label_text, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,0), 1)

                    if len(hist['centroids']) >= 5:
                        p1 = hist['centroids'][-5]
                        p2 = hist['centroids'][-1]
                        cv2.arrowedLine(frame, (cx, cy), (cx + (p2[0]-p1[0]), cy + (p2[1]-p1[1])), (0, 0, 255), 2, tipLength=0.3)

            # Dashboard Overlay
            avg_vis_speed = np.mean(current_speeds) if current_speeds else 0.0

            overlay = frame.copy()
            cv2.rectangle(overlay, (10, 10), (350, 180), (0, 0, 0), -1)
            frame = cv2.addWeighted(overlay, 0.6, frame, 0.4, 0)

            cv2.putText(frame, "TRAFFICVISION DASHBOARD", (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
            cv2.putText(frame, f"Total Detected: {len(self.counted_ids)}", (20, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
            cv2.putText(frame, f"Vehicles IN:  {self.vehicles_in}", (20, 95), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 1)
            cv2.putText(frame, f"Vehicles OUT: {self.vehicles_out}", (20, 125), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 1)
            cv2.putText(frame, f"Avg Speed (Live): {avg_vis_speed:.1f} km/h", (20, 155), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)

            out.write(frame)
            self.update_progress(frame_idx)

        cap.release()
        out.release()

        self.progress_store[self.job_id]['status'] = 'encoding'
        self.encode_video()
        self.export_analytics()
        self.progress_store[self.job_id]['status'] = 'done'

    def encode_video(self):
        cmd = [
            'ffmpeg', '-y', '-i', self.raw_output_path,
            '-vcodec', 'libx264', '-crf', '23', '-preset', 'fast',
            self.final_output_path
        ]
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        if os.path.exists(self.raw_output_path):
             os.remove(self.raw_output_path)

    def export_analytics(self):
        csv_1 = os.path.join(self.output_dir, 'per_vehicle_data.csv')
        csv_2 = os.path.join(self.output_dir, 'traffic_summary.csv')
        json_rep = os.path.join(self.output_dir, 'traffic_report.json')

        class_counts = {"car": 0, "truck": 0, "bus": 0, "motorcycle": 0, "bicycle": 0}
        speed_buckets = {"0-20": 0, "20-40": 0, "40-60": 0, "60-80": 0, "80-100": 0, "100+": 0}
        class_dwell = {"car": [], "truck": [], "bus": [], "motorcycle": [], "bicycle": []}

        max_speed = 0.0
        total_speed_all = 0.0
        per_vehicle_list = []

        # Total Vehicles equals confirmed counted_ids
        total_vehicles = len(self.counted_ids)

        with open(csv_1, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['track_id', 'vehicle_class', 'first_detected_frame', 'last_detected_frame',
                             'duration_seconds', 'avg_speed_kmh', 'max_speed_kmh', 'direction',
                             'confidence_avg', 'bounding_box_first_detection'])

            for tid, hist in self.track_history.items():
                if tid not in self.counted_ids:
                    continue

                cname = self.class_names.get(hist['class_id'], 'Unknown').lower()
                if cname in class_counts:
                    class_counts[cname] += 1

                f_frame = hist['first_seen']
                l_frame = hist['last_seen']
                dur = (l_frame - f_frame) / self.fps

                if cname in class_dwell:
                    class_dwell[cname].append(dur)

                avgs = np.mean(hist['speeds']) if hist['speeds'] else 0.0
                maxs = np.max(hist['speeds']) if hist['speeds'] else 0.0
                total_speed_all += avgs

                if maxs > max_speed: max_speed = maxs

                if avgs < 20: speed_buckets["0-20"] += 1
                elif avgs < 40: speed_buckets["20-40"] += 1
                elif avgs < 60: speed_buckets["40-60"] += 1
                elif avgs < 80: speed_buckets["60-80"] += 1
                elif avgs < 100: speed_buckets["80-100"] += 1
                else: speed_buckets["100+"] += 1

                conf_avg = np.mean(hist['conf_list']) if hist['conf_list'] else 0.0
                bbox = hist['bboxes'][0] if hist['bboxes'] else (0,0,0,0)

                writer.writerow([
                    tid, self.class_names.get(hist['class_id'], 'Unknown'), f_frame, l_frame, round(dur, 2), round(avgs, 2), round(maxs, 2),
                    hist['direction'], round(conf_avg, 3), f"{bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]}"
                ])

                per_vehicle_list.append({
                    "track_id": tid,
                    "class": cname,
                    "avg_speed": round(avgs, 2),
                    "duration_seconds": round(dur, 2),
                    "direction": hist['direction']
                })

        avg_overall = total_speed_all / total_vehicles if total_vehicles > 0 else 0.0
        duration_sec = self.total_frames / self.fps
        flow_rate = (total_vehicles / duration_sec) * 60 if duration_sec > 0 else 0.0

        summary = {
            'total_vehicles': total_vehicles,
            'avg_speed_kmh': round(avg_overall, 2),
            'peak_density': self.peak_density,
            'flow_rate_per_min': round(flow_rate, 2),
            'vehicles_in': self.vehicles_in,
            'vehicles_out': self.vehicles_out,
            'max_speed_recorded': round(max_speed, 2),
            'processing_fps': round(self.total_frames / max(0.1, (datetime.now() - self.start_time).total_seconds()), 2),
            'video_duration_seconds': round(duration_sec, 2),
            'model_used': self.config.get('model_variant'),
            'analysis_timestamp': datetime.now().isoformat()
        }

        with open(csv_2, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(list(summary.keys()) + ['cars', 'trucks', 'buses', 'motorcycles', 'bicycles'])
            writer.writerow(list(summary.values()) + [class_counts['car'], class_counts['truck'], class_counts['bus'], class_counts['motorcycle'], class_counts['bicycle']])

        class_avg_dwell = { k: round(float(np.mean(v)), 2) if v else 0.0 for k, v in class_dwell.items() }

        timeline_counts = {str(k): len(v) for k, v in self.interval_vehicles.items()}
        class_timeline = {}
        for cname in ['car', 'truck', 'bus', 'motorcycle', 'bicycle']:
            class_timeline[cname] = {str(k): len(self.class_interval_vehicles[cname][k]) for k in self.interval_vehicles.keys()}

        report_data = {
            "summary": summary,
            "class_counts": class_counts,
            "speed_distribution": speed_buckets,
            "class_avg_dwell_seconds": class_avg_dwell,
            "timeline_counts": timeline_counts,
            "class_timeline": class_timeline,
            "per_vehicle": per_vehicle_list,
            "spatial_heatmap": self.spatial_heatmap.tolist()
        }

        with open(json_rep, 'w') as f:
            json.dump(report_data, f, indent=4)

Writing processor.py


In [6]:
%%writefile app.py
import os
import uuid
import threading
from zipfile import ZipFile
from flask import Flask, request, jsonify, Response, send_file
from flask_cors import CORS

from analyzer import VideoIntelligenceAnalyzer
from processor import TrafficProcessor

app = Flask(__name__, template_folder='templates', static_folder='static')
CORS(app)
app.config['MAX_CONTENT_LENGTH'] = 500 * 1024 * 1024 # 500 MB limit

UPLOAD_FOLDER = os.path.join(os.getcwd(), 'uploads')
OUTPUT_FOLDER = os.path.join(os.getcwd(), 'outputs')
os.makedirs(UPLOAD_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Shared memory for tracking progress across threads
progress_store = {}
jobs = {}

@app.route('/')
def index():
    from flask import render_template
    return render_template('index.html')

@app.route('/upload', methods=['POST'])
def upload_video():
    if 'video' not in request.files:
        return jsonify({"error": "No video file provided"}), 400

    file = request.files['video']
    if file.filename == '':
        return jsonify({"error": "Empty file"}), 400

    job_id = str(uuid.uuid4())
    filename = job_id + "_" + file.filename.replace(" ", "_")
    filepath = os.path.join(UPLOAD_FOLDER, filename)
    file.save(filepath)

    file_size = os.path.getsize(filepath) / (1024 * 1024)

    jobs[job_id] = {
        "status": "uploaded",
        "filepath": filepath,
        "filename": filename,
        "size_mb": round(file_size, 2)
    }

    return jsonify({
        "job_id": job_id,
        "filename": filename,
        "size_mb": round(file_size, 2)
    })

@app.route('/analyze', methods=['POST'])
def analyze_video():
    data = request.json
    job_id = data.get('job_id')
    if not job_id or job_id not in jobs:
        return jsonify({"error": "Invalid job ID"}), 404

    filepath = jobs[job_id]["filepath"]
    analyzer = VideoIntelligenceAnalyzer(filepath)
    analysis = analyzer.analyze()

    if "error" in analysis:
        return jsonify(analysis), 500

    jobs[job_id]["config"] = analysis["config"]
    jobs[job_id]["status"] = "analyzed"

    return jsonify(analysis)

@app.route('/process', methods=['POST'])
def process_video():
    data = request.json
    job_id = data.get('job_id')
    custom_config = data.get('config', {}) # allow overriding auto-config

    if not job_id or job_id not in jobs:
        return jsonify({"error": "Invalid job ID"}), 404

    # Merge custom overrides into existing config
    base_config = jobs[job_id].get("config", {})
    base_config.update(custom_config)

    filepath = jobs[job_id]["filepath"]

    # Initialize progress
    progress_store[job_id] = {
        "frames_done": 0,
        "total_frames": base_config.get("total_frames", 100),
        "percent": 0,
        "current_fps": 0.0,
        "eta_seconds": 0,
        "status": "starting"
    }

    processor = TrafficProcessor(filepath, base_config, job_id, progress_store)

    # Run in background thread
    t = threading.Thread(target=processor.process, daemon=True)
    t.start()

    jobs[job_id]["status"] = "processing"

    return jsonify({"job_id": job_id, "message": "Processing started"})

@app.route('/progress/<job_id>')
def stream_progress(job_id):
    def generate():
        import time
        import json
        last_status = None
        while True:
            prog = progress_store.get(job_id)
            if prog:
                yield f"data: {json.dumps(prog)}\n\n"
                if prog.get('status') == 'done' or prog.get('status') == 'error':
                    break
            time.sleep(0.5)
    return Response(generate(), mimetype='text/event-stream')

@app.route('/results/<job_id>')
def get_results(job_id):
    json_rep = os.path.join(OUTPUT_FOLDER, job_id, 'traffic_report.json')
    if not os.path.exists(json_rep):
        return jsonify({"error": "Results not found yet"}), 404
    import json
    with open(json_rep, 'r') as f:
        data = json.load(f)
    return jsonify(data)

@app.route('/download/video/<job_id>')
def download_video(job_id):
    mp4_path = os.path.join(OUTPUT_FOLDER, job_id, 'output.mp4')
    if not os.path.exists(mp4_path):
        return jsonify({"error": "Video not found or processing failed"}), 404
    return send_file(mp4_path, as_attachment=True, download_name=f"TrafficVision_{job_id[:6]}.mp4")

@app.route('/download/csv/<job_id>')
def download_csv(job_id):
    base_dir = os.path.join(OUTPUT_FOLDER, job_id)
    zip_path = os.path.join(base_dir, 'analytics.zip')

    if not os.path.exists(zip_path):
        import zipfile
        with ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for f in ['per_vehicle_data.csv', 'traffic_summary.csv', 'traffic_report.json']:
                file_path = os.path.join(base_dir, f)
                if os.path.exists(file_path):
                    zipf.write(file_path, arcname=f)

    return send_file(zip_path, as_attachment=True, download_name=f"Analytics_{job_id[:6]}.zip")

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, threaded=True)

Writing app.py


In [7]:
%%writefile templates/index.html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>TrafficVision AI | Dashboard</title>
    <link rel="icon" href="data:image/svg+xml,<svg xmlns=%22http://www.w3.org/2000/svg%22 viewBox=%220 0 100 100%22><rect width=%22100%22 height=%22100%22 rx=%2220%22 fill=%22%230891b2%22/><path d=%22M30 50 L45 65 L70 35%22 stroke=%22white%22 stroke-width=%2210%22 fill=%22none%22 stroke-linecap=%22round%22 stroke-linejoin=%22round%22/></svg>">
    <script src="https://cdn.tailwindcss.com"></script>
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <script src="https://html2canvas.hertzen.com/dist/html2canvas.min.js"></script>
    <link href="https://fonts.googleapis.com/css2?family=Outfit:wght@300;400;500;600;700&family=JetBrains+Mono:wght@400;500;600&display=swap" rel="stylesheet">
    <style>
        body { font-family: 'Outfit', sans-serif; background-color: #0f172a; color: #f8fafc; overflow-x: hidden; }
        .mono { font-family: 'JetBrains Mono', monospace; }
        .glass { background: rgba(30, 41, 59, 0.7); backdrop-filter: blur(12px); border: 1px solid rgba(255, 255, 255, 0.1); }
        .dashboard-panel { background: #1e293b; border-radius: 12px; border: 1px inset rgba(255,255,255,0.05); padding: 24px; box-shadow: 0 4px 20px -5px rgba(0,0,0,0.4); }
        .bg-grid { background-image: radial-gradient(rgba(255, 255, 255, 0.1) 1px, transparent 1px); background-size: 20px 20px; }
        .dropzone { border: 2px dashed rgba(56, 189, 248, 0.5); transition: all 0.3s ease; }
        .dropzone.dragover { border-color: #38bdf8; background: rgba(56, 189, 248, 0.1); }
        ::-webkit-scrollbar { width: 8px; }
        ::-webkit-scrollbar-track { background: #0f172a; }
        ::-webkit-scrollbar-thumb { background: #334155; border-radius: 4px; }
        ::-webkit-scrollbar-thumb:hover { background: #475569; }
        .progress-striped { background-image: linear-gradient(45deg, rgba(255,255,255,.15) 25%, transparent 25%, transparent 50%, rgba(255,255,255,.15) 50%, rgba(255,255,255,.15) 75%, transparent 75%, transparent); background-size: 1rem 1rem; }
        @keyframes progress { from { background-position: 1rem 0; } to { background-position: 0 0; } }
        @keyframes fadeSlideUp { from { opacity: 0; transform: translateY(40px); } to { opacity: 1; transform: translateY(0); } }
        .animate-progress { animation: progress 1s linear infinite; }
        .slide-up { animation: fadeSlideUp 0.6s ease-out forwards; }
        .pulse-dot { box-shadow: 0 0 0 0 rgba(16, 185, 129, 0.7); animation: pulse 2s infinite; }
        @keyframes pulse { 0% { box-shadow: 0 0 0 0 rgba(16, 185, 129, 0.7); } 70% { box-shadow: 0 0 0 10px rgba(16, 185, 129, 0); } 100% { box-shadow: 0 0 0 0 rgba(16, 185, 129, 0); } }
        .glow-card { position: relative; overflow: hidden; }
        .glow-card::before { content:''; position: absolute; inset: -2px; background: linear-gradient(45deg, rgba(56,189,248,0.2), rgba(16,185,129,0.2)); filter: blur(10px); z-index: -1; opacity: 0; transition: opacity 0.5s; }
        .glow-card:hover::before { opacity: 1; }
    </style>
</head>
<body class="bg-grid min-h-screen text-slate-200 antialiased selection:bg-cyan-500/30">

    <!-- Header -->
    <header class="glass sticky top-0 z-50 px-6 py-4 flex items-center justify-between shadow-2xl">
        <div class="flex items-center gap-3">
            <svg class="w-8 h-8 text-cyan-400" fill="none" stroke="currentColor" viewBox="0 0 24 24"><path stroke-linecap="round" stroke-linejoin="round" stroke-width="2" d="M15 10l4.553-2.276A1 1 0 0121 8.618v6.764a1 1 0 01-1.447.894L15 14M5 18h8a2 2 0 002-2V8a2 2 0 00-2-2H5a2 2 0 00-2 2v8a2 2 0 002 2z"></path></svg>
            <div>
                <h1 class="text-xl font-bold tracking-tight text-white">TrafficVision AI</h1>
                <p class="text-xs text-slate-400">Intelligent Vehicle Detection</p>
            </div>
        </div>
        <div class="flex items-center gap-2 px-3 py-1 rounded-full bg-emerald-500/10 border border-emerald-500/20">
            <div class="w-2 h-2 rounded-full bg-emerald-500 pulse-dot"></div>
            <span class="text-xs font-semibold text-emerald-400 tracking-wide">SYSTEM LIVE</span>
        </div>
    </header>

    <main class="w-full px-4 md:px-8 xl:px-12 py-6 space-y-8 mt-2">

        <!-- SECTION 1: UPLOAD -->
        <section id="upload-section" class="glass rounded-2xl p-8 transform transition-all duration-500">
            <h2 class="text-2xl font-semibold mb-6 flex items-center gap-2">
                <span class="text-cyan-400">1.</span> Source Video
            </h2>
            <div id="dropzone" class="dropzone rounded-xl p-12 flex flex-col items-center justify-center cursor-pointer hover:bg-slate-800/50">
                <svg class="w-16 h-16 text-cyan-500 mt-2 mb-4" fill="none" stroke="currentColor" viewBox="0 0 24 24"><path stroke-linecap="round" stroke-linejoin="round" stroke-width="1.5" d="M7 16a4 4 0 01-.88-7.903A5 5 0 1115.9 6L16 6a5 5 0 011 9.9M15 13l-3-3m0 0l-3 3m3-3v12"></path></svg>
                <p class="text-lg font-medium text-slate-300">Drag & Drop video file directly here</p>
                <p class="text-sm text-slate-500 mt-2 text-center">Supports MP4, AVI, MKV up to 500MB<br>Highway, CCTV, or Drone footage</p>
                <input type="file" id="fileInput" class="hidden" accept="video/mp4,video/x-m4v,video/*">
                <button onclick="document.getElementById('fileInput').click()" class="mt-6 px-6 py-2 rounded-lg bg-cyan-600 hover:bg-cyan-500 text-white font-medium transition-colors shadow-lg shadow-cyan-500/20">Browse Files</button>
            </div>

            <div id="file-meta" class="hidden mt-6 flex flex-wrap gap-4"></div>

            <div id="analyze-btn-container" class="hidden mt-8 flex justify-end">
                <button id="analyzeBtn" class="px-8 py-3 rounded-xl bg-gradient-to-r from-blue-600 to-cyan-500 hover:from-blue-500 hover:to-cyan-400 text-white font-semibold transition-all shadow-xl shadow-cyan-500/25 flex items-center gap-2">
                    <svg class="w-5 h-5" fill="none" stroke="currentColor" viewBox="0 0 24 24"><path stroke-linecap="round" stroke-linejoin="round" stroke-width="2" d="M9.663 17h4.673M12 3v1m6.364 1.636l-.707.707M21 12h-1M4 12H3m3.343-5.657l-.707-.707m2.828 9.9a5 5 0 117.072 0l-.548.547A3.374 3.374 0 0014 18.469V19a2 2 0 11-4 0v-.531c0-.895-.356-1.754-.988-2.386l-.548-.547z"></path></svg>
                    Run V-Intelligence Analysis
                </button>
            </div>
        </section>

        <!-- SECTION 2: AUTO-CONFIG -->
        <section id="config-section" class="glass rounded-2xl p-8 hidden transform opacity-0 transition-opacity duration-700">
            <h2 class="text-2xl font-semibold mb-6 flex items-center gap-2">
                <span class="text-amber-400">2.</span> Intelligent Auto-Config
            </h2>
            <div class="bg-slate-800/50 rounded-xl p-5 mb-6 border border-slate-700/50">
                <p id="explanation-text" class="text-slate-300 text-sm leading-relaxed whitespace-pre-line mono"></p>
            </div>
            <div class="grid grid-cols-1 md:grid-cols-2 lg:grid-cols-4 gap-4" id="config-cards"></div>
            <div class="mt-8 flex justify-end">
                <button id="processBtn" class="px-8 py-3 rounded-xl bg-gradient-to-r from-emerald-600 to-teal-500 hover:from-emerald-500 hover:to-teal-400 text-white font-bold transition-all shadow-xl shadow-emerald-500/25 flex items-center gap-2 text-lg">
                    <svg class="w-6 h-6" fill="none" stroke="currentColor" viewBox="0 0 24 24"><path stroke-linecap="round" stroke-linejoin="round" stroke-width="2" d="M14.752 11.168l-3.197-2.132A1 1 0 0010 9.87v4.263a1 1 0 001.555.832l3.197-2.132a1 1 0 000-1.664z"></path></svg>
                    Start Traffic Analysis
                </button>
            </div>
        </section>

        <!-- SECTION 3: PROGRESS -->
        <section id="progress-section" class="glass rounded-2xl p-8 hidden transform opacity-0 transition-opacity duration-700">
            <h2 class="text-2xl font-semibold mb-6 flex items-center gap-2 text-white">
                <svg class="w-6 h-6 text-cyan-400 animate-spin" fill="none" viewBox="0 0 24 24"><circle class="opacity-25" cx="12" cy="12" r="10" stroke="currentColor" stroke-width="4"></circle><path class="opacity-75" fill="currentColor" d="M4 12a8 8 0 018-8V0C5.373 0 0 5.373 0 12h4zm2 5.291A7.962 7.962 0 014 12H0c0 3.042 1.135 5.824 3 7.938l3-2.647z"></path></svg>
                Processing Video
            </h2>
            <div class="mb-2 flex justify-between text-sm font-medium">
                <span id="prog-percentage" class="text-cyan-400 text-xl font-bold">0%</span>
                <span id="prog-status" class="text-slate-400 uppercase tracking-wider">Initializing Pipeline...</span>
            </div>
            <div class="h-6 w-full bg-slate-800 rounded-full overflow-hidden border border-slate-700 mb-8 p-1">
                <div id="prog-bar" class="h-full bg-gradient-to-r from-cyan-600 to-blue-500 rounded-full transition-all duration-300 ease-out progress-striped animate-progress" style="width: 0%"></div>
            </div>
            <div class="grid grid-cols-2 md:grid-cols-4 gap-4">
                <div class="bg-slate-800/30 rounded-lg p-4 border border-slate-700/50">
                    <p class="text-xs text-slate-400 uppercase">Frames</p>
                    <p class="text-xl font-medium text-white mono mt-1"><span id="stat-frames">0</span> / <span id="stat-total">0</span></p>
                </div>
                <div class="bg-slate-800/30 rounded-lg p-4 border border-slate-700/50">
                    <p class="text-xs text-slate-400 uppercase">Speed</p>
                    <p class="text-xl font-medium text-white mono mt-1"><span id="stat-fps">0.0</span> FPS</p>
                </div>
                <div class="bg-slate-800/30 rounded-lg p-4 border border-slate-700/50">
                    <p class="text-xs text-slate-400 uppercase">ETA</p>
                    <p class="text-xl font-medium text-amber-400 mono mt-1"><span id="stat-eta">--:--</span></p>
                </div>
                <div class="bg-slate-800/30 rounded-lg p-4 border border-slate-700/50">
                    <p class="text-xs text-slate-400 uppercase">Vehicles Tracked</p>
                    <p class="text-xl font-medium text-emerald-400 mono mt-1" id="stat-detected">0</p>
                </div>
            </div>
        </section>

        <!-- SECTION 4: INSIGHTS DASHBOARD -->
        <section id="results-section" class="hidden opacity-0 slide-up space-y-6 pb-20">
            <div class="flex flex-col md:flex-row items-start md:items-end justify-between mb-4 border-b border-slate-800 pb-4">
                <div>
                   <h2 class="text-3xl font-bold text-white flex items-center gap-3">
                       <svg class="w-8 h-8 text-cyan-500" fill="none" stroke="currentColor" viewBox="0 0 24 24"><path stroke-linecap="round" stroke-linejoin="round" stroke-width="2" d="M9 19v-6a2 2 0 00-2-2H5a2 2 0 00-2 2v6a2 2 0 002 2h2a2 2 0 002-2zm0 0V9a2 2 0 012-2h2a2 2 0 012 2v10m-6 0a2 2 0 002 2h2a2 2 0 002-2m0 0V5a2 2 0 012-2h2a2 2 0 012 2v14a2 2 0 01-2 2h-2a2 2 0 01-2-2z"></path></svg>
                       Traffic Intelligence Report
                   </h2>
                   <p class="text-slate-400 text-sm mt-2 font-mono" id="report-timestamp">Analysis completed at --</p>
                </div>
                <div class="flex gap-3 mt-4 md:mt-0">
                    <button id="export-png-btn" class="px-5 py-2.5 bg-slate-800 hover:bg-slate-700 rounded-lg text-white font-medium text-sm transition-colors border border-slate-600 flex items-center gap-2 shadow-lg">
                        <svg class="w-4 h-4" fill="none" stroke="currentColor" viewBox="0 0 24 24"><path stroke-linecap="round" stroke-linejoin="round" stroke-width="2" d="M4 16v1a3 3 0 003 3h10a3 3 0 003-3v-1m-4-4l-4 4m0 0l-4-4m4 4V4"></path></svg>
                        Export PNG
                    </button>
                    <a id="download-video" href="#" class="px-5 py-2.5 bg-blue-600/20 hover:bg-blue-600/40 border border-blue-500/50 rounded-lg text-blue-400 font-medium text-sm transition-colors flex items-center gap-2 shadow-lg shadow-blue-500/10">
                        <svg class="w-4 h-4" fill="none" stroke="currentColor" viewBox="0 0 24 24"><path stroke-linecap="round" stroke-linejoin="round" stroke-width="2" d="M7 4v16l13-8z"></path></svg> Video
                    </a>
                    <a id="download-csv" href="#" class="px-5 py-2.5 bg-emerald-600/20 hover:bg-emerald-600/40 border border-emerald-500/50 rounded-lg text-emerald-400 font-medium text-sm transition-colors flex items-center gap-2 shadow-lg shadow-emerald-500/10">
                        <svg class="w-4 h-4" fill="none" stroke="currentColor" viewBox="0 0 24 24"><path stroke-linecap="round" stroke-linejoin="round" stroke-width="2" d="M9 17v-2m3 2v-4m3 4v-6m2 10H7a2 2 0 01-2-2V5a2 2 0 012-2h5.586a1 1 0 01.707.293l5.414 5.414a1 1 0 01.293.707V19a2 2 0 01-2 2z"></path></svg> CSV Pack
                    </a>
                </div>
            </div>

            <div id="dashboard-capture-area" class="space-y-6 pt-2 bg-[#0f172a]" style="padding: 1rem;">
                <!-- PANEL 1: KPI SUMMARY CARDS (Top Row) -->
                <div class="grid grid-cols-1 md:grid-cols-2 lg:grid-cols-4 gap-6">
                    <div class="dashboard-panel glow-card flex items-center gap-5">
                        <div class="p-4 bg-cyan-500/10 border border-cyan-500/20 rounded-xl text-cyan-400 shadow-[0_0_15px_rgba(6,182,212,0.2)]">
                            <svg class="w-7 h-7" fill="none" stroke="currentColor" viewBox="0 0 24 24"><path stroke-linecap="round" stroke-linejoin="round" stroke-width="2" d="M8 7h12m0 0l-4-4m4 4l-4 4m0 6H4m0 0l4 4m-4-4l4-4"></path></svg>
                        </div>
                        <div>
                            <p class="text-xs text-slate-400 uppercase font-semibold tracking-wider">Total Unique Vehicles</p>
                            <p class="text-3xl font-bold text-white mt-1 mono counter" data-target="0" id="res-total">0</p>
                        </div>
                    </div>
                    <div class="dashboard-panel glow-card flex items-center gap-5">
                        <div class="p-4 bg-emerald-500/10 border border-emerald-500/20 rounded-xl text-emerald-400 shadow-[0_0_15px_rgba(16,185,129,0.2)]">
                            <svg class="w-7 h-7" fill="none" stroke="currentColor" viewBox="0 0 24 24"><path stroke-linecap="round" stroke-linejoin="round" stroke-width="2" d="M12 8v4l3 3m6-3a9 9 0 11-18 0 9 9 0 0118 0z"></path></svg>
                        </div>
                        <div>
                            <p class="text-xs text-slate-400 uppercase font-semibold tracking-wider">Average Speed</p>
                            <p class="text-3xl font-bold text-white mt-1 mono"><span class="counter" data-target="0" id="res-speed">0.0</span> <span class="text-sm font-normal text-slate-400 uppercase">km/h</span></p>
                        </div>
                    </div>
                    <div class="dashboard-panel glow-card flex items-center gap-5">
                        <div class="p-4 bg-coral-500/10 border border-red-500/20 rounded-xl text-red-400 shadow-[0_0_15px_rgba(239,68,68,0.2)]">
                            <svg class="w-7 h-7" fill="none" stroke="currentColor" viewBox="0 0 24 24"><path stroke-linecap="round" stroke-linejoin="round" stroke-width="2" d="M17.657 18.657A8 8 0 016.343 7.343S7 9 9 10c0-2 .5-5 2.986-7C14 5 16.09 5.777 17.656 7.343A7.975 7.975 0 0120 13a7.975 7.975 0 01-2.343 5.657z"></path></svg>
                        </div>
                        <div>
                            <p class="text-xs text-slate-400 uppercase font-semibold tracking-wider">Peak Density</p>
                            <p class="text-3xl font-bold text-white mt-1 mono counter" data-target="0" id="res-peak">0</p>
                        </div>
                    </div>
                    <div class="dashboard-panel glow-card flex items-center gap-5">
                        <div class="p-4 bg-violet-500/10 border border-violet-500/20 rounded-xl text-violet-400 shadow-[0_0_15px_rgba(139,92,246,0.2)]">
                            <svg class="w-7 h-7" fill="none" stroke="currentColor" viewBox="0 0 24 24"><path stroke-linecap="round" stroke-linejoin="round" stroke-width="2" d="M13 7h8m0 0v8m0-8l-8 8-4-4-6 6"></path></svg>
                        </div>
                        <div>
                            <p class="text-xs text-slate-400 uppercase font-semibold tracking-wider">Traffic Flow Rate</p>
                            <p class="text-3xl font-bold text-white mt-1 mono"><span class="counter" data-target="0" id="res-flow">0.0</span> <span class="text-sm font-normal text-slate-400 uppercase">v/min</span></p>
                        </div>
                    </div>
                </div>

                <!-- PANELS 2 & 3 -->
                <div class="grid grid-cols-1 lg:grid-cols-3 gap-6">
                    <!-- Dashboard Panel 2 -->
                    <div class="dashboard-panel lg:col-span-1 flex flex-col">
                        <h3 class="text-lg font-medium text-white mb-2">Fleet Composition</h3>
                        <p class="text-xs text-slate-400 mb-6">Proportion by vehicle class</p>
                        <div class="relative flex-1 min-h-[250px]"><canvas id="classChart"></canvas></div>
                    </div>

                    <!-- Dashboard Panel 3 -->
                    <div class="dashboard-panel lg:col-span-2 flex flex-col">
                        <h3 class="text-lg font-medium text-white mb-2">Traffic Volume Timeline</h3>
                        <p class="text-xs text-slate-400 mb-6">Unique vehicles detected per 10-second interval</p>
                        <div class="relative flex-1 min-h-[250px]"><canvas id="timelineChart"></canvas></div>
                    </div>
                </div>

                <!-- PANELS 4 & 5 -->
                <div class="grid grid-cols-1 lg:grid-cols-2 gap-6">
                    <!-- Dashboard Panel 4 -->
                    <div class="dashboard-panel flex flex-col">
                        <h3 class="text-lg font-medium text-white mb-2">Speed Distribution</h3>
                        <p class="text-xs text-slate-400 mb-6">Population density across speed buckets</p>
                        <div class="relative flex-1 min-h-[250px]"><canvas id="speedChart"></canvas></div>
                    </div>

                    <!-- Dashboard Panel 5 -->
                    <div class="dashboard-panel flex flex-col justify-center text-center">
                        <h3 class="text-lg font-medium text-white mb-2 text-left">Traffic Direction Split</h3>
                        <p class="text-xs text-slate-400 mb-8 text-left">Vehicles moving IN vs OUT over the counting line</p>

                        <div class="relative w-full h-10 bg-slate-800 rounded-full overflow-hidden flex border border-slate-700/50 shadow-inner" id="flow-bar">
                           <div id="flow-in-bar" class="h-full bg-cyan-500 transition-all duration-1000 flex items-center justify-start pl-4 text-xs font-bold text-white shadow-lg"></div>
                           <div id="flow-out-bar" class="h-full bg-amber-500 transition-all duration-1000 flex items-center justify-end pr-4 text-xs font-bold text-white shadow-lg"></div>
                        </div>
                        <div class="mt-4 mb-8">
                            <span class="text-md font-semibold text-white tracking-widest mono bg-slate-800/80 px-4 py-1.5 rounded border border-slate-700" id="flow-ratio-label">IN 50% / OUT 50%</span>
                        </div>
                        <div class="flex justify-between px-12 pb-4">
                            <div class="text-center">
                                <p class="text-4xl font-bold text-cyan-400 mono flex items-center gap-2">
                                    <svg class="w-6 h-6" fill="none" stroke="currentColor" viewBox="0 0 24 24"><path stroke-linecap="round" stroke-linejoin="round" stroke-width="3" d="M5 10l7-7m0 0l7 7m-7-7v18"></path></svg>
                                    <span id="res-in-num" class="counter" data-target="0">0</span>
                                </p>
                                <p class="text-sm text-slate-400 mt-2 uppercase font-semibold">Direction IN</p>
                            </div>
                            <div class="text-center">
                                <p class="text-4xl font-bold text-amber-400 mono flex items-center gap-2">
                                    <span id="res-out-num" class="counter" data-target="0">0</span>
                                    <svg class="w-6 h-6" fill="none" stroke="currentColor" viewBox="0 0 24 24"><path stroke-linecap="round" stroke-linejoin="round" stroke-width="3" d="M19 14l-7 7m0 0l-7-7m7 7V3"></path></svg>
                                </p>
                                <p class="text-sm text-slate-400 mt-2 uppercase font-semibold">Direction OUT</p>
                            </div>
                        </div>
                    </div>
                </div>

                <!-- PANEL 6 -->
                <div class="dashboard-panel flex flex-col">
                    <h3 class="text-lg font-medium text-white mb-2">Avg Time in Frame by Vehicle Type</h3>
                    <p class="text-xs text-slate-400 mb-6">Camera dwell time (seconds) representing flow dynamics per category</p>
                    <div class="relative min-h-[250px]"><canvas id="dwellChart"></canvas></div>
                </div>

                <!-- PANEL 7 -->
                <div class="dashboard-panel flex flex-col">
                    <div class="flex items-center justify-between mb-2">
                        <h3 class="text-lg font-medium text-white flex items-center gap-2">
                            <svg class="w-5 h-5 text-amber-500" fill="none" stroke="currentColor" viewBox="0 0 24 24"><path stroke-linecap="round" stroke-linejoin="round" stroke-width="2" d="M9 20l-5.447-2.724A1 1 0 013 16.382V5.618a1 1 0 011.447-.894L9 7m0 13l6-3m-6 3V7m6 10l4.553 2.276A1 1 0 0021 18.382V7.618a1 1 0 00-.553-.894L15 4m0 13V4m0 0L9 7"></path></svg>
                            Spatial Traffic Heatmap
                        </h3>
                        <div class="flex items-center gap-2">
                            <span class="text-xs text-slate-500 font-medium tracking-wide">CLEAR</span>
                            <div class="w-24 h-3 rounded-full bg-gradient-to-r from-emerald-500 via-yellow-500 to-red-500"></div>
                            <span class="text-xs text-slate-500 font-medium tracking-wide">CONGESTED</span>
                        </div>
                    </div>
                    <p class="text-xs text-slate-400 mb-6">2D spatial distribution of vehicle centroids mapped to the camera view.</p>
                    <div class="relative w-full border border-slate-700/50 rounded-lg overflow-hidden bg-[#0f172a] aspect-[21/9]">
                        <canvas id="heatmapCanvas" class="w-full h-full object-fill block"></canvas>
                    </div>
                </div>
            </div>
        </section>

    </main>

    <script>
        // State
        let currentJobId = null;
        let configOverride = {};

        // Colors: electric cyan, amber, coral, lime, violet
        const classColors = ['#0891b2', '#f59e0b', '#f43f5e', '#84cc16', '#8b5cf6'];
        const classLabels = ['Car', 'Truck', 'Bus', 'Motorcycle', 'Bicycle'];

        // DOM Elements
        const fileInput = document.getElementById('fileInput');
        const dropzone = document.getElementById('dropzone');
        const fileMeta = document.getElementById('file-meta');
        const analyzeBtnContainer = document.getElementById('analyze-btn-container');
        const analyzeBtn = document.getElementById('analyzeBtn');
        const configSection = document.getElementById('config-section');
        const configCards = document.getElementById('config-cards');
        const processBtn = document.getElementById('processBtn');
        const progressSection = document.getElementById('progress-section');
        const resultsSection = document.getElementById('results-section');

        // Setup Chart Defaults
        Chart.defaults.color = '#cbd5e1';
        Chart.defaults.font.family = "'Outfit', sans-serif";
        const animConfig = { duration: 1000, easing: 'easeInOutQuart' };

        // Handlers
        dropzone.addEventListener('dragover', e => { e.preventDefault(); dropzone.classList.add('dragover'); });
        dropzone.addEventListener('dragleave', e => { e.preventDefault(); dropzone.classList.remove('dragover'); });
        dropzone.addEventListener('drop', e => { e.preventDefault(); dropzone.classList.remove('dragover'); if(e.dataTransfer.files.length) handleFile(e.dataTransfer.files[0]); });
        fileInput.addEventListener('change', e => { if(e.target.files.length) handleFile(e.target.files[0]); });

        function handleFile(file) {
            const formData = new FormData();
            formData.append('video', file);
            dropzone.innerHTML = `<svg class="w-12 h-12 text-cyan-500 animate-spin mx-auto mb-4" fill="none" viewBox="0 0 24 24"><circle class="opacity-25" cx="12" cy="12" r="10" stroke="currentColor" stroke-width="4"></circle><path class="opacity-75" fill="currentColor" d="M4 12a8 8 0 018-8V0C5.373 0 0 5.373 0 12h4zm2 5.291A7.962 7.962 0 014 12H0c0 3.042 1.135 5.824 3 7.938l3-2.647z"></path></svg><p>Uploading ${file.name}...</p>`;

            fetch('/upload', { method: 'POST', body: formData })
            .then(res => res.json())
            .then(data => {
                currentJobId = data.job_id;
                dropzone.classList.add('hidden');
                fileMeta.innerHTML = `<div class="flex items-center gap-2 px-4 py-2 rounded-full bg-slate-800 border border-slate-700 text-sm mono text-slate-300"><svg class="w-4 h-4 text-cyan-400" fill="none" stroke="currentColor" viewBox="0 0 24 24"><path stroke-linecap="round" stroke-linejoin="round" stroke-width="2" d="M7 4v16M17 4v16M3 8h4m10 0h4M3 12h18M3 16h4m10 0h4M4 20h16a1 1 0 001-1V5a1 1 0 00-1-1H4a1 1 0 00-1 1v14a1 1 0 001 1z"></path></svg> ${data.filename.substring(37)}</div><div class="flex items-center gap-2 px-4 py-2 rounded-full bg-slate-800 border border-slate-700 text-sm mono text-slate-300"><svg class="w-4 h-4 text-emerald-400" fill="none" stroke="currentColor" viewBox="0 0 24 24"><path stroke-linecap="round" stroke-linejoin="round" stroke-width="2" d="M4 7v10c0 2 1 3 3 3h10c2 0 3-1 3-3V7c0-2-1-3-3-3H7c-2 0-3 1-3 3z"></path><path stroke-linecap="round" stroke-linejoin="round" stroke-width="2" d="M9 17v-4a3 3 0 016 0v4"></path></svg> ${data.size_mb} MB</div>`;
                fileMeta.classList.remove('hidden');
                analyzeBtnContainer.classList.remove('hidden');
            }).catch(err => alert("Upload failed: " + err));
        }

        analyzeBtn.addEventListener('click', () => {
            analyzeBtn.disabled = true;
            analyzeBtn.innerHTML = '<svg class="w-5 h-5 animate-spin mr-2" fill="none" viewBox="0 0 24 24"><circle class="opacity-25" cx="12" cy="12" r="10" stroke="currentColor" stroke-width="4"></circle><path class="opacity-75" fill="currentColor" d="M4 12a8 8 0 018-8V0C5.373 0 0 5.373 0 12h4zm2 5.291A7.962 7.962 0 014 12H0c0 3.042 1.135 5.824 3 7.938l3-2.647z"></path></svg> Analyzing...';

            fetch('/analyze', { method: 'POST', headers: {'Content-Type': 'application/json'}, body: JSON.stringify({job_id: currentJobId}) })
            .then(res => res.json()).then(data => {
                document.getElementById('explanation-text').innerText = data.explanation;
                const renderCard = (title, val) => `<div class="bg-slate-800/30 rounded-lg p-4 border border-slate-700/50"><p class="text-xs text-slate-400 uppercase">${title}</p><p class="text-lg font-medium text-white mono mt-1">${val}</p></div>`;
                configCards.innerHTML = `${renderCard('Resolution', data.config.resolution + ` (${data.config.width}x${data.config.height})`)}${renderCard('Model Payload', data.config.model_variant)}${renderCard('Conf. Threshold', data.config.conf_threshold)}${renderCard('Processing Skip', data.config.frame_skip + ' frames')}`;
                configSection.classList.remove('hidden');
                setTimeout(() => configSection.classList.remove('opacity-0'), 100);
                analyzeBtnContainer.classList.add('hidden');
            });
        });

        processBtn.addEventListener('click', () => {
            configSection.classList.add('hidden');
            progressSection.classList.remove('hidden');
            setTimeout(() => progressSection.classList.remove('opacity-0'), 100);

            fetch('/process', { method: 'POST', headers: {'Content-Type': 'application/json'}, body: JSON.stringify({job_id: currentJobId, config: configOverride}) });
            pollProgress();
        });

        function pollProgress() {
            const evtSource = new EventSource(`/progress/${currentJobId}`);
            evtSource.onmessage = function(event) {
                const p = JSON.parse(event.data);
                document.getElementById('prog-bar').style.width = p.percent + '%';
                document.getElementById('prog-percentage').innerText = p.percent + '%';
                document.getElementById('prog-status').innerText = p.status;
                document.getElementById('stat-frames').innerText = p.frames_done;
                document.getElementById('stat-total').innerText = p.total_frames;
                document.getElementById('stat-fps').innerText = p.current_fps.toFixed(1);

                const mins = Math.floor(p.eta_seconds / 60);
                const secs = p.eta_seconds % 60;
                document.getElementById('stat-eta').innerText = `${mins}:${secs.toString().padStart(2, '0')}`;
                document.getElementById('stat-detected').innerText = p.vehicles_detected || 0;

                if (p.status === 'done') {
                    evtSource.close();
                    document.getElementById('prog-status').innerText = 'COMPLETED';
                    document.getElementById('prog-bar').classList.remove('animate-progress');
                    setTimeout(fetchResults, 1000);
                }
            };
        }

        function extractDataArrayForClassOrder(dictData) {
            return [dictData.car || 0, dictData.truck || 0, dictData.bus || 0, dictData.motorcycle || 0, dictData.bicycle || 0];
        }

        function animateCounters() {
            document.querySelectorAll('.counter').forEach(counter => {
                const target = parseFloat(counter.getAttribute('data-target')) || 0;
                const duration = 1200;
                const startTime = performance.now();
                const isFloat = counter.id === 'res-speed' || counter.id === 'res-flow';

                const updateCounter = (currentTime) => {
                    const elapsedTime = currentTime - startTime;
                    if (elapsedTime < duration) {
                        const progress = Math.min(1, elapsedTime / duration);
                        // easeOutQuad
                        const easeOut = progress * (2 - progress);
                        const val = target * easeOut;
                        counter.innerText = isFloat ? val.toFixed(1) : Math.round(val);
                        requestAnimationFrame(updateCounter);
                    } else {
                        counter.innerText = isFloat ? target.toFixed(1) : target;
                    }
                };
                requestAnimationFrame(updateCounter);
            });
        }

        function fetchResults() {
            progressSection.classList.add('hidden');

            fetch(`/results/${currentJobId}`).then(res => res.json()).then(data => {

                // Add downloads
                document.getElementById('download-video').href = `/download/video/${currentJobId}`;
                document.getElementById('download-csv').href = `/download/csv/${currentJobId}`;

                // Set text data
                document.getElementById('report-timestamp').innerText = "Analysis completed at " + new Date(data.summary.analysis_timestamp).toLocaleString();

                // Setup Counters
                document.getElementById('res-total').setAttribute('data-target', data.summary.total_vehicles);
                document.getElementById('res-speed').setAttribute('data-target', data.summary.avg_speed_kmh);
                document.getElementById('res-peak').setAttribute('data-target', data.summary.peak_density);
                document.getElementById('res-flow').setAttribute('data-target', data.summary.flow_rate_per_min);
                document.getElementById('res-in-num').setAttribute('data-target', data.summary.vehicles_in);
                document.getElementById('res-out-num').setAttribute('data-target', data.summary.vehicles_out);

                // Setup Flow Bar Split
                const totalIn = data.summary.vehicles_in || 0;
                const totalOut = data.summary.vehicles_out || 0;
                const totalDir = totalIn + totalOut || 1;
                const inPct = Math.round((totalIn / totalDir) * 100);
                const outPct = 100 - inPct;

                // Reveal section to start animations
                resultsSection.classList.remove('hidden');

                // Wait a moment for layout to settle before animating
                setTimeout(() => {
                    document.getElementById('flow-in-bar').style.width = inPct + '%';
                    document.getElementById('flow-out-bar').style.width = outPct + '%';
                    document.getElementById('flow-ratio-label').innerText = `IN ${inPct}% / OUT ${outPct}%`;
                    animateCounters();
                }, 100);

                // Build Panel 2: Fleet Composition
                const classData = extractDataArrayForClassOrder(data.class_counts);
                new Chart(document.getElementById('classChart'), {
                    type: 'doughnut',
                    data: { labels: classLabels, datasets: [{ data: classData, backgroundColor: classColors, borderWidth: 0 }] },
                    options: { responsive: true, maintainAspectRatio: false, animation: animConfig, plugins: { legend: { position: 'bottom', labels: { boxWidth: 12 } } }, cutout: '75%' }
                });

                // Build Panel 3: Volume Timeline
                const rawKeys = Object.keys(data.timeline_counts).map(Number).sort((a,b)=>a-b);
                const timelineLabels = rawKeys.map(k => `${k*10}s`);
                const timelineVals = rawKeys.map(k => data.timeline_counts[String(k)]);

                let peakIdx = 0, maxV = -1;
                timelineVals.forEach((v,i) => { if(v>maxV){ maxV=v; peakIdx=i; } });

                new Chart(document.getElementById('timelineChart'), {
                    type: 'line',
                    data: {
                        labels: timelineLabels,
                        datasets: [{
                            label: 'Vehicles',
                            data: timelineVals,
                            borderColor: '#38bdf8', backgroundColor: 'rgba(56, 189, 248, 0.15)',
                            borderWidth: 2, fill: true, tension: 0.4,
                            pointBackgroundColor: '#0ea5e9', pointRadius: 3
                        }]
                    },
                    options: {
                        responsive: true, maintainAspectRatio: false, animation: animConfig,
                        plugins: { legend: { display: false } },
                        scales: { x: { grid: { display: false } }, y: { beginAtZero: true, grid: { color: 'rgba(255,255,255,0.05)' } } }
                    }
                });

                // Build Panel 4: Speed Distribution
                const speedBuckets = ["0-20", "20-40", "40-60", "60-80", "80-100", "100+"];
                const speedVals = speedBuckets.map(b => data.speed_distribution[b] || 0);
                new Chart(document.getElementById('speedChart'), {
                    type: 'bar',
                    data: {
                        labels: ['0-20', '21-40', '41-60', '61-80', '81-100', '100+'],
                        datasets: [{
                            label: 'Vehicles',
                            data: speedVals,
                            backgroundColor: (ctx) => {
                                const map = ['#22c55e', '#84cc16', '#eab308', '#f97316', '#ef4444', '#b91c1c'];
                                return map[ctx.dataIndex];
                            },
                            borderRadius: 4
                        }]
                    },
                    options: { responsive: true, maintainAspectRatio: false, animation: animConfig, plugins: { legend: { display: false } }, scales: { x: { grid: { display: false } }, y: { beginAtZero: true, grid: { color: 'rgba(255,255,255,0.05)' } } } }
                });

                // Build Panel 6: Dwell Time
                const dwellVals = extractDataArrayForClassOrder(data.class_avg_dwell_seconds);
                new Chart(document.getElementById('dwellChart'), {
                    type: 'bar',
                    data: { labels: classLabels, datasets: [{ label: 'Avg Sec in Frame', data: dwellVals, backgroundColor: classColors, borderRadius: 4 }] },
                    options: { indexAxis: 'y', responsive: true, maintainAspectRatio: false, animation: animConfig, plugins: { legend: { display: false } }, scales: { x: { beginAtZero: true, grid: { color: 'rgba(255,255,255,0.05)' } }, y: { grid: { display: false } } } }
                });

                // Build Panel 7: Heatmap (Spatial)
                drawHeatmap(data.spatial_heatmap);
            });
        }

        // Heatmap Component Draw
        function drawHeatmap(spatialData) {
            const canvas = document.getElementById('heatmapCanvas');
            const ctx = canvas.getContext('2d');

            if (!spatialData || !spatialData.length) return;

            const rows = spatialData.length;
            const cols = spatialData[0].length;

            // Set canvas resolution to match aspect ratio but higher quality
            canvas.width = cols * 20;
            canvas.height = rows * 20;

            const cellW = canvas.width / cols;
            const cellH = canvas.height / rows;

            let maxVal = 0;
            for(let r=0; r<rows; r++){
                for(let c=0; c<cols; c++){
                    if(spatialData[r][c] > maxVal) maxVal = spatialData[r][c];
                }
            }

            ctx.clearRect(0, 0, canvas.width, canvas.height);

            for(let r=0; r<rows; r++){
                for(let c=0; c<cols; c++){
                    const val = spatialData[r][c];

                    if (val === 0) {
                        continue; // Transparent/bg
                    }

                    // Add tiny baseline to make points visible even for single appearances if scale is huge
                    const ratio = Math.max(0.05, Math.pow(val / maxVal, 0.7));

                    let R, G, B;
                    if (ratio < 0.5) {
                        const pct = ratio * 2;
                        R = Math.round(16 + pct * (234 - 16));
                        G = Math.round(185 + pct * (179 - 185));
                        B = Math.round(129 + pct * (8 - 129));
                    } else {
                        const pct = (ratio - 0.5) * 2;
                        R = Math.round(234 + pct * (239 - 234));
                        G = Math.round(179 + pct * (68 - 179));
                        B = Math.round(8 + pct * (68 - 8));
                    }

                    ctx.fillStyle = `rgba(${R}, ${G}, ${B}, ${0.5 + (ratio * 0.5)})`;
                    ctx.fillRect(c * cellW, r * cellH, cellW + 0.5, cellH + 0.5);
                }
            }
        }

        // HTML2Canvas Export
        document.getElementById('export-png-btn').addEventListener('click', () => {
            const btn = document.getElementById('export-png-btn');
            const ogText = btn.innerHTML;
            btn.innerText = 'Creating PNG...';

            html2canvas(document.getElementById('dashboard-capture-area'), {
                backgroundColor: '#0f172a',
                scale: 1.5,
                logging: false
            }).then(canvas => {
                 const link = document.createElement('a');
                 link.download = `Traffic_Insight_Report.png`;
                 link.href = canvas.toDataURL('image/png');
                 link.click();
                 btn.innerHTML = ogText;
            });
        });
    </script>
</body>
</html>

Writing templates/index.html


In [ ]:
# Authenticate Ngrok (Get auth token from https://dashboard.ngrok.com)
NGROK_AUTH_TOKEN = "TOKEN"  # <-- Replace with your token
!ngrok config add-authtoken $NGROK_AUTH_TOKEN

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [9]:
# START SERVER
from pyngrok import ngrok
import subprocess
import threading
import time

# Kill any existing tunnels
ngrok.kill()

# Start Flask app in background
def run_flask():
    subprocess.run(["python", "app.py"], cwd="/content")

t = threading.Thread(target=run_flask, daemon=True)
t.start()

time.sleep(3)

# Open Ngrok Tunnel
public_url = ngrok.connect(5000)
print("="*60)
print("✅ TrafficVision AI Backend is LIVE!")
print(f"🌐 Access the Web UI here: {public_url}")
print("="*60)

✅ TrafficVision AI Backend is LIVE!
🌐 Access the Web UI here: NgrokTunnel: "https://emmanuel-unadapted-pitchily.ngrok-free.dev" -> "http://localhost:5000"
